# YoungNAKO · Machine Learning · Hands-on 2
#### Thomas Küstner, University Hospital of Tübingen
## Beyond tables: segmentation, foundation models, embeddings & CLIP

**Thursday 05.11.2026, Block 2 (≈ 30 min)** · Thomas Küstner, MIDAS.lab, University Hospital Tübingen

Three applications on the same abdominal MRI data (CHAOS, T1 in-phase/opposed-phase = 2-point Dixon):

| Part | Application | Method | ≈ time |
|---|---|---|---|
| **A** | **Segmentation → image-derived phenotypes** (organ volumes, liver fat) | TotalSegmentator MRI, a pre-trained nnU-Net | 10 min |
| **B** | **Classification & regression** from foundation-model **embeddings** | DINOv2 + logistic/ridge regression ("linear probing") | 10 min |
| **C** | **Contrastive learning: CLIP** (zero-shot) and train your own mini-CLIP | BiomedCLIP, PyTorch | 10 min |

Cells marked **✏️ TODO** contain a blank `___` to fill in. Training a full network from scratch takes hours; the code for that (nnU-Net, CLIP fine-tuning, MedSAM2, your own NAKO Dixon data) is in **Notebook 4, for home**.

> ⚙️ `Runtime → Change runtime type → T4 GPU` is strongly recommended for this notebook.

In [ ]:
#@title 📦 Setup: installs TotalSegmentator, OpenCLIP, UMAP (≈ 2 min) { display-mode: "form" }
# Install the requirements into the running kernel: requirements.txt of the repo if it is next to
# this notebook (local / cloned repo), otherwise (Colab opened via link) this notebook's packages.
import os
_req = next((p for p in ("requirements.txt", "../requirements.txt") if os.path.isfile(p)), None)
get_ipython().run_line_magic("pip", f"install -q -r {_req}" if _req else
                             "install -q numpy pandas matplotlib scipy scikit-learn torch pillow pydicom SimpleITK transformers TotalSegmentator open_clip_torch umap-learn")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU = DEVICE == "cuda"
torch.manual_seed(0); np.random.seed(0)
print("Running on:", DEVICE)

In [ ]:
#@title 🔧 Helper functions: download & load the CHAOS MRI data (just run this cell) { display-mode: "form" }
# CHAOS challenge data (Kavur et al., Med Image Anal 2021), Zenodo record 3431873, licence CC BY-NC-SA 4.0.
# We use the 20 MR training cases, T1-DUAL sequence: in-phase (IP) and opposed-phase (OP) images
# plus expert masks of liver, right kidney, left kidney and spleen.
import os, io, glob, time, zipfile, urllib.request, shutil
from concurrent.futures import ThreadPoolExecutor
import numpy as np

CHAOS_URL = "https://zenodo.org/records/3431873/files/CHAOS_Train_Sets.zip?download=1"
# Colab: /content/chaos · locally: ~/yn_data/chaos (shared by all notebooks, outside Dropbox/git); override with YN_CHAOS_ROOT
CHAOS_DIR = os.environ.get("YN_CHAOS_ROOT", "/content/chaos" if os.path.isdir("/content") else os.path.expanduser("~/yn_data/chaos"))
LABELS = {1: "liver", 2: "kidney_right", 3: "kidney_left", 4: "spleen"}


def _get(url, start, end, tries=6):
    """Bytes start..end of a web file, retrying dropped connections (Zenodo is sometimes slow/flaky)."""
    for k in range(tries):
        try:
            req = urllib.request.Request(url, headers={"Range": f"bytes={start}-{end}"})
            with urllib.request.urlopen(req, timeout=60) as r:
                if r.status != 206:
                    raise IOError("server ignores byte ranges")
                data = r.read()
            if len(data) == end - start + 1:
                return data
        except Exception as e:
            if "ignores byte ranges" in str(e) or k == tries - 1:
                raise
        time.sleep(2 * (k + 1))
    raise IOError(f"incomplete download of bytes {start}-{end}")


class _RemoteZip(io.RawIOBase):
    """Seekable read-only view of a zip on a web server: only the byte ranges that are read get downloaded."""
    def __init__(self, url):
        self.url, self.pos, self.chunks = url, 0, {}
        with urllib.request.urlopen(urllib.request.Request(url, method="HEAD"), timeout=60) as r:
            self.size = int(r.headers["Content-Length"])
    def readable(self): return True
    def seekable(self): return True
    def tell(self): return self.pos
    def seek(self, off, whence=0):
        self.pos = (off, self.pos + off, self.size + off)[whence]
        return self.pos
    def prefetch(self, start, end):
        self.chunks[start] = _get(self.url, start, min(end, self.size - 1))
    def read(self, n=-1):
        n = self.size - self.pos if n is None or n < 0 else min(n, self.size - self.pos)
        if n <= 0:
            return b""
        for s, d in self.chunks.items():
            if s <= self.pos and self.pos + n <= s + len(d):
                out = d[self.pos - s:self.pos - s + n]
                break
        else:
            out = _get(self.url, self.pos, self.pos + n - 1)
        self.pos += n
        return out


def _is_mr_t1dual(name):
    name = name.replace("\\", "/")
    return "/MR/" in name and "/T1DUAL/" in name and not name.endswith("/")


def _download_mr_only(dest):
    """Fetch only the MR T1-DUAL files (≈ 65 MB) out of the 0.9 GB zip, in ~1 MB blocks, 8 in parallel."""
    rz = _RemoteZip(CHAOS_URL)
    z = zipfile.ZipFile(rz)
    members = sorted((i for i in z.infolist() if _is_mr_t1dual(i.filename)), key=lambda i: i.header_offset)
    spans = []                                    # merge neighbouring files into larger download blocks
    for i in members:
        s, e = i.header_offset, i.header_offset + 30 + len(i.filename.encode()) + len(i.extra) + i.compress_size + 1024
        if spans and s - spans[-1][1] < (1 << 18) and e - spans[-1][0] < (4 << 20):
            spans[-1][1] = e
        else:
            spans.append([s, e])
    print(f"Downloading {len(members)} MR files (≈ {sum(i.compress_size for i in members) / 1e6:.0f} MB) from Zenodo ...")
    done = [0]
    def fetch(span):
        rz.prefetch(*span)
        done[0] += 1
        if done[0] % 10 == 0 or done[0] == len(spans):
            print(f"  {done[0]}/{len(spans)} blocks")
    with ThreadPoolExecutor(8) as ex:
        list(ex.map(fetch, spans))
    z.extractall(dest, members)


def _download_full(dest):
    """Fallback: the whole zip (≈ 0.9 GB), resuming after dropped connections."""
    zpath = dest.rstrip("/") + "_download.zip"
    with urllib.request.urlopen(urllib.request.Request(CHAOS_URL, method="HEAD"), timeout=60) as r:
        size = int(r.headers["Content-Length"])
    print("Downloading the full CHAOS training zip (≈ 0.9 GB) ...")
    for k in range(20):
        have = os.path.getsize(zpath) if os.path.exists(zpath) else 0
        if have >= size:
            break
        try:
            req = urllib.request.Request(CHAOS_URL, headers={"Range": f"bytes={have}-"} if have else {})
            with urllib.request.urlopen(req, timeout=60) as r, open(zpath, "ab" if r.status == 206 else "wb") as f:
                shutil.copyfileobj(r, f, length=1 << 22)
        except Exception as e:
            print(f"  connection dropped ({type(e).__name__}), resuming at {have / 1e6:.0f} MB ...")
            time.sleep(5)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(dest, [m for m in z.namelist() if _is_mr_t1dual(m)])
    os.remove(zpath)


def download_chaos(dest=CHAOS_DIR, force=False):
    """Download and unpack the 20 MR cases (T1-DUAL) of the CHAOS training set.
    Safe to re-run: complete data is reused, leftovers of an interrupted download are deleted first."""
    marker = os.path.join(dest, ".download_complete")
    if os.path.exists(marker) and not force:
        print("CHAOS data already available in", dest)
        return dest
    if os.path.isdir(dest):                       # incomplete/old download -> start clean (else slice counts/shapes mismatch)
        shutil.rmtree(dest)
    os.makedirs(dest)
    try:
        _download_mr_only(dest)
    except Exception as e:
        print(f"Partial download not possible ({type(e).__name__}: {str(e)[:100]})")
        shutil.rmtree(dest); os.makedirs(dest)
        _download_full(dest)
    open(marker, "w").close()
    print("Done:", len(glob.glob(os.path.join(dest, "**", "T1DUAL"), recursive=True)), "MR cases in", dest)
    return dest


def _png_to_labels(png):
    """CHAOS ground-truth PNGs encode organs as grey values: liver 63, R kidney 126, L kidney 189, spleen 252."""
    lab = np.zeros(png.shape, np.uint8)
    lab[(png >= 55) & (png <= 70)] = 1
    lab[(png >= 110) & (png <= 135)] = 2
    lab[(png >= 175) & (png <= 200)] = 3
    lab[(png >= 240)] = 4
    return lab


def load_case(case_dir):
    """Load one CHAOS T1-DUAL case.
    Returns dict with 'ip', 'op' (float32, z×y×x), 'mask' (uint8, z×y×x),
    'spacing' (x, y, z in mm), 'origin', 'direction' (for building a 3D image), 'id'."""
    import pydicom
    from PIL import Image
    t1 = os.path.join(case_dir, "T1DUAL")
    ip_files = sorted(glob.glob(os.path.join(t1, "DICOM_anon", "InPhase", "*.dcm")))
    op_files = sorted(glob.glob(os.path.join(t1, "DICOM_anon", "OutPhase", "*.dcm")))
    gt_files = sorted(glob.glob(os.path.join(t1, "Ground", "*.png")))
    n = min(len(ip_files), len(op_files), len(gt_files))
    if not (len(ip_files) == len(op_files) == len(gt_files)):
        print(f"  note: case {os.path.basename(case_dir)} has {len(ip_files)}/{len(op_files)}/{len(gt_files)} IP/OP/GT slices, using {n}")
    ip_ds = [pydicom.dcmread(f) for f in ip_files[:n]]
    op_ds = [pydicom.dcmread(f) for f in op_files[:n]]
    gt = [_png_to_labels(np.array(Image.open(f).convert("L"))) for f in gt_files[:n]]
    # geometry from DICOM header -> sort slices along the slice normal
    iop = np.array(ip_ds[0].ImageOrientationPatient, float)
    r, c = iop[:3], iop[3:]
    nrm = np.cross(r, c)
    pos = np.array([np.dot(np.array(d.ImagePositionPatient, float), nrm) for d in ip_ds])
    order = np.argsort(pos)
    ip = np.stack([ip_ds[i].pixel_array.astype(np.float32) for i in order])
    op = np.stack([op_ds[i].pixel_array.astype(np.float32) for i in order])
    mask = np.stack([gt[i] for i in order])
    dz = float(np.median(np.diff(pos[order]))) if n > 1 else float(ip_ds[0].SliceThickness)
    ps = [float(v) for v in ip_ds[0].PixelSpacing]  # [row spacing, column spacing]
    direction = (r[0], c[0], nrm[0], r[1], c[1], nrm[1], r[2], c[2], nrm[2])
    origin = tuple(float(v) for v in ip_ds[order[0]].ImagePositionPatient)
    return dict(ip=ip, op=op, mask=mask, spacing=(ps[1], ps[0], abs(dz)), origin=origin,
                direction=direction, id=os.path.basename(case_dir))


def load_all_cases(root=CHAOS_DIR):
    case_dirs = sorted({os.path.dirname(p) for p in glob.glob(os.path.join(root, "**", "MR", "*", "T1DUAL"), recursive=True)},
                       key=lambda s: int(os.path.basename(s)) if os.path.basename(s).isdigit() else s)
    cases = [load_case(d) for d in case_dirs]
    print(f"Loaded {len(cases)} cases, {sum(c['ip'].shape[0] for c in cases)} slices per contrast")
    return cases


def to_sitk(vol, case):
    """Wrap a z×y×x numpy array into a SimpleITK image with the case geometry."""
    import SimpleITK as sitk
    img = sitk.GetImageFromArray(vol)
    img.SetSpacing(case["spacing"]); img.SetOrigin(case["origin"]); img.SetDirection(case["direction"])
    return img


def _norm(vol):
    lo, hi = np.percentile(vol, 1), np.percentile(vol, 99.5)
    return np.clip((vol - lo) / (hi - lo + 1e-6), 0, 1).astype(np.float32)


def make_slice_dataset(cases, size=128, min_kidney_px=50):
    """Turn the 3D cases into a 2D slice dataset.
    X_ip, X_op : (N, size, size) images scaled to [0, 1] (per volume)
    y_kidney   : 1 if at least one kidney is visible in the slice
    z_rel      : relative slice position in the volume (0 = lowest slice, 1 = highest)
    patient    : case id (for patient-level splitting!)
    organs     : list of visible organs per slice (used for captions)"""
    import torch, torch.nn.functional as F
    X_ip, X_op, M, yk, zr, pid, organs = [], [], [], [], [], [], []
    for c in cases:
        ipn, opn = _norm(c["ip"]), _norm(c["op"])
        nz = ipn.shape[0]
        for z in range(nz):
            m = c["mask"][z]
            X_ip.append(ipn[z]); X_op.append(opn[z]); M.append(m)
            k = int(((m == 2) | (m == 3)).sum() >= min_kidney_px)
            yk.append(k); zr.append(z / max(nz - 1, 1)); pid.append(c["id"])
            organs.append([LABELS[l] for l in (1, 2, 3, 4) if (m == l).sum() >= min_kidney_px])
    def rs(a, mode):
        t = torch.from_numpy(np.stack(a)[:, None].astype(np.float32))
        t = F.interpolate(t, size=(size, size), mode=mode, **({} if mode == "nearest" else {"align_corners": False}))
        return t[:, 0].numpy()
    X_ip, X_op = rs(X_ip, "bilinear"), rs(X_op, "bilinear")
    M = rs(M, "nearest").astype(np.uint8)
    return dict(X_ip=X_ip, X_op=X_op, mask=M, y_kidney=np.array(yk), z_rel=np.array(zr, np.float32),
                patient=np.array(pid), organs=organs)


def patient_split(patients, n_val=3, n_test=4, seed=0):
    """Split by PATIENT (never by slice!) into train/val/test index arrays."""
    ids = np.array(sorted(set(patients), key=lambda s: int(s) if str(s).isdigit() else s))
    rng = np.random.default_rng(seed); rng.shuffle(ids)
    test_ids, val_ids, train_ids = ids[:n_test], ids[n_test:n_test + n_val], ids[n_test + n_val:]
    idx = lambda s: np.where(np.isin(patients, s))[0]
    return idx(train_ids), idx(val_ids), idx(test_ids)

In [ ]:
download_chaos()
cases = load_all_cases()

---
# Part A: Segmentation with a foundation model → image-derived phenotypes

**TotalSegmentator MRI** (Akinci D'Antonoli et al., *Radiology* 2025) is an nnU-Net trained on > 600 MRI and CT scans to segment ~50 structures, **sequence-independent**. We use it *as is*, without any training ("inference only").

The CHAOS experts' masks let us check how good it is on data it has never seen.

In [ ]:
import SimpleITK as sitk
from totalsegmentator.python_api import totalsegmentator
from totalsegmentator.map_to_binary import class_map

ROIS = list(LABELS.values())   # liver, kidney_right, kidney_left, spleen

def run_totalseg(case, out="seg.nii.gz"):
    """Run TotalSegmentator MRI on the in-phase image of one case, return a label map in CHAOS coding."""
    sitk.WriteImage(to_sitk(case["ip"], case), "ip.nii.gz")
    attempts = [dict(roi_subset=ROIS, fast=not GPU), dict(fast=not GPU), dict()]
    for kw in attempts:                      # fall back gracefully if an option is not supported
        try:
            totalsegmentator("ip.nii.gz", out, task="total_mr", ml=True, quiet=True, **kw); break
        except Exception as e:
            print("  option", kw, "not available here (", type(e).__name__, ") -> retrying with fewer options")
    seg = sitk.GetArrayFromImage(sitk.ReadImage(out))
    name2id = {v: k for k, v in class_map["total_mr"].items()}
    pred = np.zeros(seg.shape, np.uint8)
    for lab, name in LABELS.items():
        pred[seg == name2id[name]] = lab
    return pred

case = cases[0]
pred = run_totalseg(case)
print("case", case["id"], "| volume", case["ip"].shape, "| organs found:", [LABELS[l] for l in np.unique(pred) if l])

In [ ]:
def dice(a, b):
    # ✏️ TODO: Dice = 2·|A ∩ B| / (|A| + |B|)   (hint: (a & b).sum() is the overlap)
    return ___ / (a.sum() + b.sum() + 1e-9)

gt = case["mask"]
print({name: round(dice(pred == l, gt == l), 3) for l, name in LABELS.items()})

z = int(np.argmax(((gt == 2) | (gt == 3)).sum((1, 2))))     # slice with the most kidney
fig, ax = plt.subplots(1, 3, figsize=(13, 4.5))
cmap = plt.matplotlib.colors.ListedColormap(["none", "tab:orange", "tab:green", "tab:cyan", "tab:purple"])
ax[0].imshow(case["ip"][z], cmap="gray"); ax[0].set_title("in-phase")
ax[1].imshow(case["ip"][z], cmap="gray"); ax[1].imshow(gt[z], cmap=cmap, vmin=0, vmax=4, alpha=0.5); ax[1].set_title("expert (CHAOS)")
ax[2].imshow(case["ip"][z], cmap="gray"); ax[2].imshow(pred[z], cmap=cmap, vmin=0, vmax=4, alpha=0.5); ax[2].set_title("TotalSegmentator MRI")
for a in ax: a.axis("off")
plt.tight_layout()

### A2 · From masks to numbers an epidemiologist can use
* **Organ volume** = number of voxels × voxel volume
* **Liver fat**: with in-phase (IP) and opposed-phase (OP) images, fat and water signals add up in IP and cancel in OP. The **signal fat fraction** is

$$ FF \approx \frac{IP - OP}{2 \cdot IP} $$

(valid for FF < 50 %; NAKO's Dixon protocol also gives separate fat/water images, see Notebook 4).

In [ ]:
from scipy.ndimage import binary_erosion

def phenotypes(case, pred):
    vox_ml = np.prod(case["spacing"]) / 1000.0
    row = {"case": case["id"]}
    for l, name in LABELS.items():
        row[f"{name}_ml"] = (pred == l).sum() * vox_ml
    liver = binary_erosion(pred == 1, iterations=2)            # stay away from the organ border
    ip, op = case["ip"][liver], case["op"][liver]
    # ✏️ TODO: signal fat fraction (in %) from in-phase and opposed-phase signal
    ff = ___ * 100
    row["liver_fat_%"] = float(np.median(ff)) if ff.size else np.nan
    return row

pd.DataFrame([phenotypes(case, pred)]).round(1)

### A3 · Scale it up: a small phenotype table
On a GPU each case takes a few seconds; on CPU about a minute. This is exactly how image-derived phenotypes are extracted for the 30,000 NAKO MRI participants (just with more organs and quality control).

In [ ]:
N_CASES = 8 if GPU else 2            # increase if you have time
rows, dices = [], []
for c in cases[:N_CASES]:
    p = run_totalseg(c)
    rows.append(phenotypes(c, p))
    dices.append({"case": c["id"], **{n: dice(p == l, c["mask"] == l) for l, n in LABELS.items()}})
pheno = pd.DataFrame(rows); dsc = pd.DataFrame(dices)
display(pheno.round(1)); display(dsc.round(3))
print("mean Dice per organ:"); print(dsc.drop(columns="case").mean().round(3))

**❓ Questions:** Which organ is segmented worst, and why might that be (size, contrast, partial volume)? Is a Dice of 0.9 "good enough" to use the liver volume as an exposure variable in an epidemiological model?

---
# Part B: Foundation-model embeddings for classification & regression

A **foundation model** has been pre-trained on huge amounts of data and turns every image into a feature vector (*embedding*). We use **DINOv2** (Meta, self-supervised on 142 M natural images, never trained on MRI!) and put a simple linear model on top: **linear probing**.

In [ ]:
from transformers import AutoImageProcessor, AutoModel

data = make_slice_dataset(cases, size=224)
X, yk, zr, pat = data["X_ip"], data["y_kidney"], data["z_rel"], data["patient"]
tr, va, te = patient_split(pat, n_val=3, n_test=4, seed=0)     # same split as in Hands-on 1

proc = AutoImageProcessor.from_pretrained("facebook/dinov2-small")
dino = AutoModel.from_pretrained("facebook/dinov2-small").to(DEVICE).eval()

@torch.no_grad()
def embed_dino(X, bs=64):
    out = []
    for i in range(0, len(X), bs):
        imgs = [np.repeat((x * 255).astype(np.uint8)[..., None], 3, -1) for x in X[i:i + bs]]
        inp = proc(images=imgs, return_tensors="pt").to(DEVICE)
        out.append(dino(**inp).pooler_output.float().cpu())
    return torch.cat(out).numpy()

E = embed_dino(X)
print("embeddings:", E.shape, "→ every slice is now a point in a", E.shape[1], "-dimensional space")

### B1 · Look at the embedding space with UMAP

In [ ]:
import umap
Z2 = umap.UMAP(n_neighbors=15, min_dist=0.2, random_state=0).fit_transform(E)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
ax[0].scatter(*Z2.T, c=yk, cmap="coolwarm", s=8); ax[0].set_title("coloured by: kidney visible")
sc = ax[1].scatter(*Z2.T, c=zr, cmap="viridis", s=8); ax[1].set_title("coloured by: slice position"); plt.colorbar(sc, ax=ax[1])
codes = pd.factorize(pat)[0]
ax[2].scatter(*Z2.T, c=codes, cmap="tab20", s=8); ax[2].set_title("coloured by: patient")
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout()

**❓ Question:** Do the slices cluster by anatomy or by *patient*? What would it mean for a downstream model if they cluster by patient (or by scanner / study centre)?

### B2 · Classification: kidney visible? (linear probe)

In [ ]:
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, mean_absolute_error

# ✏️ TODO: train on the training slices only  (hint: index arrays are tr, va, te)
probe = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000)).fit(E[___], yk[___])
auc_dino = roc_auc_score(yk[te], probe.predict_proba(E[te])[:, 1])
print(f"DINOv2 + logistic regression, TEST AUC = {auc_dino:.3f}   (compare with your CNN from Hands-on 1)")

### B3 · Regression: where in the body is this slice?
Predicting the anatomical position of a slice ("body-part regression") is a classic self-supervised task. Here it is a ridge regression on the embeddings.

In [ ]:
# ✏️ TODO: choose a regularisation strength for the ridge regression (e.g. 1, 10, 100)
reg = make_pipeline(StandardScaler(), Ridge(alpha=___)).fit(E[tr], zr[tr])
zp = np.clip(reg.predict(E[te]), 0, 1)
print(f"TEST mean absolute error = {mean_absolute_error(zr[te], zp)*100:.1f} % of the volume height")
plt.figure(figsize=(4.5, 4)); plt.scatter(zr[te], zp, s=10, c=codes[te], cmap="tab20")
plt.plot([0, 1], [0, 1], "k--", lw=1); plt.xlabel("true relative position"); plt.ylabel("predicted"); plt.title("body-part regression")
plt.show()

**Optional experiment (data efficiency):** train the probe with only **2 patients** (`tr_small = tr[np.isin(pat[tr], np.unique(pat[tr])[:2])]`). How much does the AUC drop compared with the CNN trained from scratch?

---
# Part C: Contrastive learning & CLIP

**CLIP** learns from image–text pairs: matching pairs are pulled together in a shared embedding space, all other pairs in the batch are pushed apart (contrastive / InfoNCE loss). Afterwards you can classify images with **text prompts only** (*zero-shot*).

We use **BiomedCLIP** (Microsoft, trained on 15 M figure–caption pairs from PubMed Central).

In [ ]:
import open_clip
from PIL import Image
HF_ID = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
clip_model, clip_pre = open_clip.create_model_from_pretrained(HF_ID)
clip_tok = open_clip.get_tokenizer(HF_ID)
clip_model = clip_model.to(DEVICE).eval()

@torch.no_grad()
def clip_images(X, bs=64):
    out = []
    for i in range(0, len(X), bs):
        ims = torch.stack([clip_pre(Image.fromarray((x * 255).astype(np.uint8)).convert("RGB")) for x in X[i:i + bs]])
        out.append(torch.nn.functional.normalize(clip_model.encode_image(ims.to(DEVICE)).float(), dim=-1).cpu())
    return torch.cat(out)

@torch.no_grad()
def clip_texts(prompts):
    t = clip_tok(prompts, context_length=256).to(DEVICE)
    return torch.nn.functional.normalize(clip_model.encode_text(t).float(), dim=-1).cpu()

C_img = clip_images(X)
print("BiomedCLIP image embeddings:", tuple(C_img.shape))

In [ ]:
# ✏️ TODO: write two prompts: [description of the POSITIVE class, description of the NEGATIVE class]
#    e.g. ["an axial MRI image of the abdomen at the level of the kidneys", "an axial MRI image of the upper abdomen, no kidneys"]
prompts = ___
T = clip_texts(prompts)
probs = (100 * C_img @ T.T).softmax(-1)[:, 0].numpy()        # probability of the first prompt
print(f"ZERO-SHOT BiomedCLIP, TEST AUC = {roc_auc_score(yk[te], probs[te]):.3f}   (no training at all!)")

probe_clip = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000)).fit(C_img[tr].numpy(), yk[tr])
print(f"BiomedCLIP + linear probe,   TEST AUC = {roc_auc_score(yk[te], probe_clip.predict_proba(C_img[te].numpy())[:, 1]):.3f}")
print(f"DINOv2 + linear probe,       TEST AUC = {auc_dino:.3f}")

**❓ Questions:** Try other prompts. How sensitive is the zero-shot result to the wording? Why is zero-shot weaker than the linear probe here? (Think: how often do PubMed captions describe *axial T1 Dixon slices at kidney level*?)

### C2 · Train your own mini-CLIP in 2 minutes (if time)
We automatically write a caption for every slice from the expert masks, e.g. *"opposed-phase mri of the middle abdomen showing liver and kidney right"*, and train a tiny image encoder (CNN) and text encoder (word embeddings) **contrastively**.

In [ ]:
import torch.nn as nn, torch.nn.functional as F

def caption(phase, z, organs):
    region = "upper" if z > 0.66 else "middle" if z > 0.33 else "lower"
    org = " and ".join(o.replace("_", " ") for o in organs) if organs else "no labelled organ"
    return f"{phase} mri of the {region} abdomen showing {org}"

small = lambda A: F.interpolate(torch.tensor(A[:, None]), size=(96, 96), mode="bilinear", align_corners=False)
imgs = torch.cat([small(data["X_ip"]), small(data["X_op"])])
caps = [caption("in-phase", z, o) for z, o in zip(zr, data["organs"])] + [caption("opposed-phase", z, o) for z, o in zip(zr, data["organs"])]
pat2 = np.concatenate([pat, pat])
vocab = {w: i + 1 for i, w in enumerate(sorted({w for c in caps for w in c.split()}))}
def tokenize(texts, L=16):
    out = torch.zeros(len(texts), L, dtype=torch.long)
    for i, t in enumerate(texts):
        ids = [vocab.get(w, 0) for w in t.lower().split()][:L]; out[i, :len(ids)] = torch.tensor(ids)
    return out
toks = tokenize(caps)
print(len(caps), "image–caption pairs, vocabulary:", len(vocab), "words\nexample:", caps[len(caps)//2 + 10])

class ImageEncoder(nn.Module):
    def __init__(self, d=64):
        super().__init__()
        ch, layers, c_in = (16, 32, 64, 64), [], 1
        for c in ch: layers += [nn.Conv2d(c_in, c, 3, padding=1), nn.BatchNorm2d(c), nn.ReLU(), nn.MaxPool2d(2)]; c_in = c
        self.net = nn.Sequential(*layers, nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c_in, d))
    def forward(self, x): return F.normalize(self.net(x), dim=-1)

class TextEncoder(nn.Module):
    def __init__(self, V, d=64):
        super().__init__(); self.emb = nn.Embedding(V + 1, d, padding_idx=0); self.proj = nn.Linear(d, d)
    def forward(self, t):
        m = (t > 0).float().unsqueeze(-1)
        return F.normalize(self.proj((self.emb(t) * m).sum(1) / m.sum(1).clamp(min=1)), dim=-1)

In [ ]:
torch.manual_seed(0)
tr_ids = set(pat[tr]); train_mask = np.isin(pat2, list(tr_ids))
img_enc, txt_enc = ImageEncoder().to(DEVICE), TextEncoder(len(vocab)).to(DEVICE)
log_temp = nn.Parameter(torch.tensor(np.log(1 / 0.07), dtype=torch.float32, device=DEVICE))
opt = torch.optim.Adam(list(img_enc.parameters()) + list(txt_enc.parameters()) + [log_temp], lr=1e-3)
I_tr, T_tr = imgs[train_mask], toks[train_mask]

for ep in range(25):
    perm = torch.randperm(len(I_tr)); tot = 0
    for i in range(0, len(I_tr), 128):
        idx = perm[i:i + 128]
        fi, ft = img_enc(I_tr[idx].to(DEVICE)), txt_enc(T_tr[idx].to(DEVICE))
        logits = log_temp.exp() * fi @ ft.T                     # similarity of every image with every caption
        target = torch.arange(len(idx), device=DEVICE)          # the matching pair is on the diagonal
        # ✏️ TODO: symmetric contrastive loss = mean of image→text and text→image cross-entropy
        loss = ___
        opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(idx)
    if ep % 5 == 4: print(f"epoch {ep+1}: contrastive loss {tot/len(I_tr):.3f}")

In [ ]:
# Text → image retrieval on the TEST patients (never seen during training)
test_mask = np.isin(pat2, list(set(pat[te])))
I_te = imgs[test_mask]
with torch.no_grad():
    F_te = img_enc.eval()(I_te.to(DEVICE)).cpu()

# ✏️ TODO: type your own query (use words from the captions: in-phase/opposed-phase, upper/middle/lower, liver, spleen, kidney left/right)
query = ___
with torch.no_grad():
    q = txt_enc.eval()(tokenize([query]).to(DEVICE)).cpu()
top = (F_te @ q.T).squeeze(1).argsort(descending=True)[:6]
fig, ax = plt.subplots(1, 6, figsize=(15, 3))
for a, k in zip(ax, top):
    a.imshow(I_te[k, 0], cmap="gray"); a.axis("off"); a.set_title(np.array(caps)[test_mask][k][:38] + "…", fontsize=7)
plt.suptitle(f"query: {query}", fontsize=10); plt.show()

**❓ Question:** Our captions were generated from labels, so every batch contains many *identical* captions that the loss treats as "wrong" pairs. Why is that a problem, and how do real CLIP datasets avoid it?

---
### ✅ Summary
* **Pre-trained segmentation models** (TotalSegmentator, MRSegmentator, nnU-Net) turn MRI into **image-derived phenotypes**, i.e. new variables for epidemiology. Check their quality (Dice) first!
* **Foundation-model embeddings** + a linear model give strong classifiers/regressors with little data.
* **CLIP** links images and text through **contrastive learning** and enables zero-shot use, but prompts and domain matter.

🏠 **At home:** Notebook 4 trains nnU-Net on CHAOS, fine-tunes BiomedCLIP, points to MedSAM2 and shows how to run all of this on **your own NAKO Dixon data**, locally, nothing has to be shared.

➡️ Next block: **Does it actually work?** Evaluation and pitfalls.